<a href="https://colab.research.google.com/github/Abhishek-M-K2005/YieldVoyager/blob/model_abhi/backend/risk_engine/yieldvoyager3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install xgboost shap pandas numpy scikit-learn requests tqdm

In [2]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import requests
import shap
import xgboost as xgb
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler


In [3]:
protocols = [
    "aave",
    "curve-dex",
    "makerdao",
    "compound-finance",
    "lido",
    "uniswap",
]

In [4]:
def fetch_protocol(protocol):
   url = f"https://api.llama.fi/protocol/{protocol}"
   response = requests.get(url)
   # if the response is successful, then, extract the json from the response and return the data. else, return empty list.
   if response.ok:
       try:
           data = response.json()
           return data
       except requests.exceptions.JSONDecodeError:
           print(f"Warning: Could not decode JSON for protocol {protocol}. Response content: {response.text}")
           return {}
   else:
       print(f"Warning: Failed to fetch data for protocol {protocol}. Status code: {response.status_code}")
       return {}


In [5]:
def build_dataset(protocol_name, protocol_id):
  data = fetch_protocol(protocol_name)
  rows = []

  listed_at = data.get("listedAt")
  # if the protocol specific Total value locked by date json object is not there, then, return empty dataframe.
  """
  'format like:'
  ........
  "chainTvls":{
        "Ethereum-borrowed":{
          "tvl":[
            {
              "date":1538006400,
              "totalLiquidityUSD":3390
            },
              ......
          ],
          ...
        },
        ....
    },
  ......
  """
  if "chainTvls" not in data:
    return pd.DataFrame()
  for chain in data["chainTvls"]:
    tvl_data = data["chainTvls"][chain].get("tvl", [])
    if len(tvl_data) < 10:
      continue

    current_util_ratio = np.random.uniform(0.20, 0.70)

    for i in range(7, len(tvl_data)):

      latest = tvl_data[i]["totalLiquidityUSD"]
      tvl_24h = tvl_data[i - 1]["totalLiquidityUSD"]
      tvl_7d = tvl_data[i - 7]["totalLiquidityUSD"]

      if tvl_7d == 0 or tvl_24h == 0:
        continue

      tvl_change_24h = (latest - tvl_24h) / tvl_24h
      tvl_change_7d = (latest - tvl_7d) / tvl_7d
      liquidity_depth = latest



      borrowed_key = f"{chain}-borrowed"
      borrowed = data.get("currentChainTvls", {}).get(borrowed_key, 0)
      utilisation_ratio = borrowed / liquidity_depth if liquidity_depth != 0 else 0;

      oracle_price_std = abs(tvl_change_24h) * 10

      liquidation_spike_ratio = abs(tvl_change_24h) * 10

      if listed_at:
        protocol_age_days = int((tvl_data[i]["date"] - listed_at)/86400)
      else:
        protocol_age_days = int((tvl_data[i]["date"] - tvl_data[0]["date"])/86400)

      protocol_age_days = max(1, protocol_age_days)

      audit_count = 1 + (protocol_age_days // 180)
      audit_count = min(5, audit_count)

      util_noise = np.random.normal(0, 0.02)
      current_util_ratio += util_noise;

      current_util_ratio = max(0.05, min(0.95, current_util_ratio))
      utilisation_ratio = current_util_ratio

      base_volatility = np.random.uniform(0.01, 0.05);
      oracle_price_std = base_volatility + (abs(tvl_change_24h) * np.random.uniform(0.5, 1.5))

      if tvl_change_24h < -0.05:
          liquidation_spike_ratio = abs(tvl_change_24h) * np.random.uniform(2.0, 4.0)
      else:
          liquidation_spike_ratio = np.random.uniform(0.0, 0.01)
      instability_label = 1 if tvl_change_7d < -0.30 else 0

      rows.append([
                protocol_id,
                protocol_name,
                chain,
                tvl_data[i]["date"],
                tvl_change_24h,
                tvl_change_7d,
                liquidity_depth,
                utilisation_ratio,
                oracle_price_std,
                liquidation_spike_ratio,
                protocol_age_days,
                audit_count,
                instability_label
            ])
  columns = [
      "protocol_id",
      "protocol_name",
      "chain",
      "date",
      "tvl_change_24h",
      "tvl_change_7d",
      "liquidity_depth",
      "utilisation_ratio",
      "oracle_price_std",
      "liquidation_spike_ratio",
      "protocol_age_days",
      "audit_count",
      "instability_label"
  ]

  return pd.DataFrame(rows, columns=columns )


In [ ]:

all_dfs=[]
for idx, protocol in enumerate(tqdm(protocols)):
  df_p = build_dataset(protocol, idx)
  if not df_p.empty:
    all_dfs.append(df_p)
df = pd.concat(all_dfs, ignore_index=True)
print("Total rows:", len(df))
df

100%|██████████| 6/6 [00:27<00:00,  4.66s/it]

Total rows: 131580


,protocol_id,protocol_name,chain,date,tvl_change_24h,tvl_change_7d,liquidity_depth,utilisation_ratio,oracle_price_std,liquidation_spike_ratio,protocol_age_days,audit_count,instability_label
0,0,aave,Ethereum-staking,1607558400,-0.047319,0.002764,244769550,0.326839,0.088657,0.004764,7,1,0
1,0,aave,Ethereum-staking,1607644800,-0.045431,-0.039229,233649315,0.293184,0.086170,0.006439,8,1,0
2,0,aave,Ethereum-staking,1607731200,-0.100323,-0.160264,210208955,0.257932,0.146804,0.323717,9,1,0
3,0,aave,Ethereum-staking,1607817600,0.125564,-0.067544,236603705,0.258997,0.106234,0.007859,10,1,0
4,0,aave,Ethereum-staking,1607904000,0.024082,-0.053742,242301482,0.269643,0.053166,0.006233,11,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
131575,5,uniswap,Soneium,1773446400,-0.000573,0.019964,2669128,0.761694,0.012681,0.006770,409,3,0
131576,5,uniswap,Soneium,1773532800,0.000244,0.017057,2669780,0.766058,0.017168,0.001378,410,3,0
131577,5,uniswap,Soneium,1773619200,0.004001,0.019683,2680463,0.807817,0.021212,0.006189,411,3,0
131578,5,uniswap,Soneium,1773705600,0.009272,0.027522,2705317,0.787621,0.034331,0.007123,412,3,0


In [ ]:
df.to_csv('data.csv')

In [ ]:
from google.colab import files

files.download('data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df1 = df.sort_values(["protocol_id", "date"]).reset_index(drop = True)

In [ ]:
print((df['instability_label'] == 1).sum())

3369


experiment 2


In [6]:


# extract data from defi-llama yields api.
def fetch_structural_risk():
    url = "https://yields.llama.fi/pools"
    response = requests.get(url)
    pools = response.json().get("data", [])

    rows = []
    seen_projects = set()
    for pool in pools:
        project_name = pool.get("project")
        if project_name not in seen_projects:
            rows.append({
                "protocol_name": project_name.lower() if project_name else "",
                "impermanent_loss_risk": 1 if pool.get("ilRisk") == "yes" else 0,
                "is_single_exposure": 1 if pool.get("exposure") == "single" else 0,
            })
            seen_projects.add(project_name)
    return pd.DataFrame(rows)

# build dataset using data.
def build_dataset_2(protocol_name, protocol_id):
    # (Assuming fetch_protocol is already defined in your notebook)
    data = fetch_protocol(protocol_name)
    rows = []


    if "chainTvls" not in data:
      return pd.DataFrame()

    for chain in data["chainTvls"]:
        tvl_data = data["chainTvls"][chain].get("tvl", [])
        if len(tvl_data) < 10: continue

        current_util_ratio = np.random.uniform(0.20, 0.70)

        for i in range(7, len(tvl_data)):
            latest = tvl_data[i]["totalLiquidityUSD"]
            tvl_24h = tvl_data[i - 1]["totalLiquidityUSD"]
            tvl_7d = tvl_data[i - 7]["totalLiquidityUSD"]

            if tvl_7d == 0 or tvl_24h == 0: continue

            tvl_change_24h = (latest - tvl_24h) / tvl_24h
            tvl_change_7d = (latest - tvl_7d) / tvl_7d


            age_days = int((tvl_data[i]["date"] - (tvl_data[0]["date"]))/86400)
            age_days = max(1, age_days)

            # Trying to add non-uniformity in the data to make it a realistic data, which can be learnable by machine learning model.

            current_util_ratio = max(0.05, min(0.95, current_util_ratio + np.random.normal(0, 0.02)))
            oracle_price_std = np.random.uniform(0.01, 0.05) + (abs(tvl_change_24h) * np.random.uniform(0.5, 1.5))
            liquidation_spike = abs(tvl_change_24h) * np.random.uniform(2.0, 4.0) if tvl_change_24h < -0.05 else np.random.uniform(0.0, 0.01)

            # Risk label evaluation method using fixed aggregation.
            tvl_risk = min(40, (max(0, -tvl_change_7d) / 0.30) * 40) # max contribution to risk: 40%
            util_risk = min(30, ((current_util_ratio - 0.60) / 0.40) * 30) if current_util_ratio > 0.60 else 0 # max contribution to risk: 30%
            vol_risk = min(15, (oracle_price_std / 0.20) * 15) # max contribution to risk: 15%
            liq_risk = min(15, (liquidation_spike / 0.10) * 15) # max contribution to risk: 15%

            risk_label = round((tvl_risk + util_risk + vol_risk + liq_risk) / 100.0, 4)

            rows.append([
                protocol_id, protocol_name, chain, tvl_data[i]["date"],
                tvl_change_24h, tvl_change_7d, current_util_ratio,
                oracle_price_std, liquidation_spike, age_days, risk_label
            ])

    columns = ["protocol_id", "protocol_name", "chain", "date", "tvl_change_24h", "tvl_change_7d", "utilisation_ratio", "oracle_price_std", "liquidation_spike_ratio", "protocol_age_days", "risk_label"]
    return pd.DataFrame(rows, columns=columns)

# 3. Execution & Merge
all_dfs = [build_dataset_2(p, i) for i, p in enumerate(tqdm(protocols))]
historical_df = pd.concat(all_dfs, ignore_index=True)
structural_df = fetch_structural_risk()
df = pd.merge(historical_df, structural_df, on="protocol_name", how="left")
df.fillna(0, inplace=True) # Clean up missing structural data

100%|██████████| 6/6 [00:04<00:00,  1.29it/s]


In [7]:

all_dfs=[]
for idx, protocol in enumerate(tqdm(protocols)):
  df_p = build_dataset_2(protocol, idx)
  if not df_p.empty:
    all_dfs.append(df_p)
df = pd.concat(all_dfs, ignore_index=True)
print("Total rows:", len(df))
df

100%|██████████| 6/6 [00:04<00:00,  1.21it/s]

Total rows: 132307


,protocol_id,protocol_name,chain,date,tvl_change_24h,tvl_change_7d,utilisation_ratio,oracle_price_std,liquidation_spike_ratio,protocol_age_days,risk_label
0,0,aave,Ethereum-staking,1607558400,-0.047319,0.002764,0.282278,0.101493,0.002091,7,0.0793
1,0,aave,Ethereum-staking,1607644800,-0.045431,-0.039229,0.288741,0.070931,0.008991,8,0.1190
2,0,aave,Ethereum-staking,1607731200,-0.100323,-0.160264,0.251593,0.120350,0.296822,9,0.4539
3,0,aave,Ethereum-staking,1607817600,0.125564,-0.067544,0.250706,0.167069,0.006458,10,0.2250
4,0,aave,Ethereum-staking,1607904000,0.024082,-0.053742,0.264365,0.082435,0.008663,11,0.1465
...,...,...,...,...,...,...,...,...,...,...,...
132302,5,uniswap,Soneium,1773878400,0.000225,0.023631,0.522705,0.038608,0.005599,414,0.0374
132303,5,uniswap,Soneium,1773964800,0.004185,0.017076,0.513420,0.028646,0.001787,415,0.0242
132304,5,uniswap,Soneium,1774051200,0.001965,0.019660,0.535172,0.036816,0.001568,416,0.0300
132305,5,uniswap,Soneium,1774137600,-0.003595,0.015746,0.559048,0.033415,0.009021,417,0.0386


In [22]:
bins = [i / 20 for i in range(0, 21)]
print(df['risk_label'].value_counts(bins=bins, sort=False).sort_index())


(-0.001, 0.05]    32104
(0.05, 0.1]       29123
(0.1, 0.15]       16460
(0.15, 0.2]       13627
(0.2, 0.25]       11676
(0.25, 0.3]        9659
(0.3, 0.35]        5828
(0.35, 0.4]        3538
(0.4, 0.45]        3383
(0.45, 0.5]        2044
(0.5, 0.55]        1191
(0.55, 0.6]         938
(0.6, 0.65]         850
(0.65, 0.7]        1173
(0.7, 0.75]         242
(0.75, 0.8]         160
(0.8, 0.85]         137
(0.85, 0.9]          94
(0.9, 0.95]          69
(0.95, 1.0]          11
Name: count, dtype: int64


In [ ]:
df.to_csv('risk.csv')

KeyboardInterrupt: 

In [ ]:
from google.colab import files

files.download('risk.csv')

In [23]:
!pip install tf

In [24]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [25]:
features = [
    "tvl_change_24h",
    "tvl_change_7d",
    "utilisation_ratio",
    "oracle_price_std",
    "liquidation_spike_ratio",
    "protocol_age_days"
]

target = "risk_label"

In [34]:
import pandas as pd
from sklearn.utils import class_weight

df['risk_bin'] = pd.cut(df["risk_label"], bins = 10, labels = False)

bin_count = df['risk_bin'].value_counts()
total_samples= len(df)

bin_weights = {bin_id: total_samples / count for bin_id, count in bin_count.items()}

sample_weights_series = df["risk_bin"].map(bin_weights) # Keep as a Series for indexed access

In [81]:
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

def create_sequences(data, sample_weights_series, window_size=7):
  X, y, weights = [], [], []
  for _, group in data.groupby(['protocol_id', 'chain']):
    group_values = group[features].values
    labels = group[target].values
    # Get the corresponding weights for this group using its index
    group_sample_weights = sample_weights_series.loc[group.index].values

    if len(group_values) > window_size:
      for i in range(window_size, len(group_values)):
        X.append(group_values[i-window_size:i, :])
        y.append(labels[i])
        weights.append(group_sample_weights[i]) # Append corresponding weight

  return np.array(X), np.array(y), np.array(weights)

# Call create_sequences to get X, y, and aligned sample weights
X, y, sample_weights_aligned = create_sequences(df, sample_weights_series, window_size = 7)

In [77]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.losses import Huber

for cpu

In [39]:
def build_rnn(window_size=10, num_features=6):
  inputs = Input(shape=(window_size, num_features))
  lstm_layer = LSTM(64, name="risk_embedding")(inputs)
  x = Dropout(0.3)(lstm_layer)
  x = Dense(16, activation="relu")(x)

  output_score= Dense(1, activation='sigmoid', name="risk_score")(x)
  model = Model(inputs=inputs, outputs=output_score)
  model.compile(optimizer="adam", loss=Huber(delta=0.5), metrics=["mae", tf.keras.metrics.RootMeanSquaredError()])

  encoder=Model(inputs=inputs, outputs=lstm_layer)

  return model, encoder

model, encoder = build_rnn()
model.summary()


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 10, 6)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ risk_embedding (LSTM)           │ (None, 64)             │        18,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ risk_score (Dense)              │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,233 (75.13 KB)

 Trainable params: 19,233 (75.13 KB)

 Non-trainable params: 0 (0.00 B)

32, 0.2, 16, 1: 0.0656

In [40]:
X_train, X_test, y_train, y_test, weights_train, weights_test = train_test_split(X, y, sample_weights_aligned, test_size=0.2, random_state=42)
history = model.fit(
    X_train, y_train,
    epochs=25,
    sample_weight=weights_train,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/25
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - loss: 0.2346 - mae: 0.1980 - root_mean_squared_error: 0.2239 - val_loss: 0.2068 - val_mae: 0.1636 - val_root_mean_squared_error: 0.1943
Epoch 2/25
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - loss: 0.2211 - mae: 0.1786 - root_mean_squared_error: 0.2034 - val_loss: 0.1929 - val_mae: 0.2042 - val_root_mean_squared_error: 0.2221
Epoch 3/25
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - loss: 0.2186 - mae: 0.1801 - root_mean_squared_error: 0.2030 - val_loss: 0.1869 - val_mae: 0.1662 - val_root_mean_squared_error: 0.1863
Epoch 4/25
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - loss: 0.2138 - mae: 0.1769 - root_mean_squared_error: 0.2007 - val_loss: 0.2047 - val_mae: 0.1717 - val_root_mean_squared_error: 0.1965
Epoch 5/25
2617/2617 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - loss: 0.2142 - mae: 0.1752 - root_mean_squared_error: 0.1992 - val_loss: 0.1875 - val_mae: 0.1759 - val_root_mean_squared_error: 0.1938
Epoch 6/25
2617/2617 ━━━━━━━━━━━━━━

for gpu

In [132]:
from tensorflow.keras.mixed_precision import set_global_policy
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, GlobalAveragePooling1D
set_global_policy('mixed_float16')

def build_rnn_gpu(window_size=7, num_features=6 ):
  inputs = tf.keras.Input(shape = (window_size, num_features))

  x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
  x = Dropout(0.3)(x)
  x = LSTM(32, return_sequences=True)(x)
  risk_embedding = GlobalAveragePooling1D(name="risk_embedding")(x)
  x = tf.keras.layers.Dropout(0.3)(risk_embedding)
  x = tf.keras.layers.Dense(32, activation=None)(x)
  x = tf.keras.layers.LeakyReLU(alpha=0.01)(x)
  output_score= tf.keras.layers.Dense(1, activation = "sigmoid", dtype = 'float32', name = "risk_score")(x)

  model = tf.keras.Model(inputs = inputs, outputs = output_score)

  model.compile(
        optimizer=tf.keras.optimizers.Lion(learning_rate=1e-3),
        loss=tf.keras.losses.Huber(delta=1),
        metrics=["mae", tf.keras.metrics.RootMeanSquaredError()],
        jit_compile=True
    )
  encoder = tf.keras.Model(inputs=inputs, outputs=risk_embedding)
  return model, encoder

In [133]:
X_train, X_test, y_train, y_test, weights_train, weights_test = train_test_split(
    X, y, sample_weights_aligned, test_size=0.2, random_state=42
    )
def prepare_dataset(X_data, y_data, w_data, batch_size = 128):
  dataset = tf.data.Dataset.from_tensor_slices((X_data, y_data, w_data))
  dataset = dataset.shuffle(len(X_data)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
  return dataset

train_ds = prepare_dataset(X_train, y_train, weights_train, batch_size=128)
val_ds = prepare_dataset(X_test, y_test, weights_test, batch_size=128)


In [134]:
model, encoder = build_rnn_gpu()

# Early stopping prevents wasting GPU cycles if the model stops improving
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
]

history = model.fit(
    train_ds,
    epochs=25,
    validation_data=val_ds,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/25
821/821 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - loss: 0.2463 - mae: 0.1961 - root_mean_squared_error: 0.2222 - val_loss: 0.2104 - val_mae: 0.1801 - val_root_mean_squared_error: 0.2081
Epoch 2/25
821/821 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 0.2305 - mae: 0.1841 - root_mean_squared_error: 0.2093 - val_loss: 0.2081 - val_mae: 0.1807 - val_root_mean_squared_error: 0.2008
Epoch 3/25
821/821 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - loss: 0.2252 - mae: 0.1779 - root_mean_squared_error: 0.2019 - val_loss: 0.2065 - val_mae: 0.1867 - val_root_mean_squared_error: 0.2114
Epoch 4/25
821/821 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.2157 - mae: 0.1740 - root_mean_squared_error: 0.1979 - val_loss: 0.1908 - val_mae: 0.1513 - val_root_mean_squared_error: 0.1743
Epoch 5/25
821/821 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.2165 - mae: 0.1741 - root_mean_squared_error: 0.1984 - val_loss: 0.1957 - val_mae: 0.1509 - val_root_mean_squared_error: 0.1796
Epoch 6/25
821/821 ━━━━━━━━━━━━━━━━━━━━ 9s 

In [ ]:
model.save('/Users/abhishekmarutikarennavar/Documents/YieldVoyager/YieldVoyager/backend/risk_engine/models/rnn_risk_models.h5')
encoder.save('/Users/abhishekmarutikarennavar/Documents/YieldVoyager/YieldVoyager/backend/risk_engine/models/risk_encoder.h5')

In [ ]:
from google.colab import files

files.download('/Users/abhishekmarutikarennavar/Documents/YieldVoyager/YieldVoyager/backend/risk_engine/models/rnn_risk_models.h5')
files.download('/Users/abhishekmarutikarennavar/Documents/YieldVoyager/YieldVoyager/backend/risk_engine/models/risk_encoder.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:
# Test: Get a vector for a single protocol sequence
sample_input = X_test[0:1]
risk_vector = encoder.predict(sample_input)

print(f"Vector ready for ChromaDB: {risk_vector[0]}")
print(f"Vector Dimension: {len(risk_vector[0])}") # Should be 32

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step
Vector ready for ChromaDB: [ 0.05823   0.10315  -0.06366  -0.085    -0.1066   -0.0856    0.0861
 -0.09216  -0.03772   0.0342   -0.1497   -0.04593   0.05185   0.1229
  0.05142  -0.10754   0.1566   -0.08325   0.08606   0.0834    0.131
 -0.1691    0.1902   -0.05914   0.1497   -0.0529   -0.27      0.006653
  0.06995  -0.00837   0.05472  -0.2993   -0.1543   -0.09924  -0.1296
  0.05573   0.00388  -0.00249   0.1758   -0.07336  -0.0585    0.10144
  0.07434  -0.1448    0.2922   -0.0825   -0.08154  -0.1742   -0.1519
 -0.1122   -0.1597    0.01507  -0.1208   -0.1168   -0.0489   -0.1366
 -0.09393  -0.0722   -0.143     0.03305  -0.1895   -0.07367   0.1562
  0.03223 ]
Vector Dimension: 64
